# Ejercicio: Diseña e Implementa un Sistema Multi-Agente

En el notebook anterior aprendiste los 4 patrones de arquitectura multi-agente:

| Patrón | Cuándo |
|---|---|
| Orquestador LLM | El orden es dinámico, el modelo decide |
| Sequential | Pipeline fijo, cada paso depende del anterior |
| Parallel | Tareas independientes, la velocidad importa |
| Loop | Refinamiento iterativo hasta criterio de calidad |

Este ejercicio tiene **dos partes**:

- **Parte 1 — Diseño:** dado un problema, propones la arquitectura antes de escribir código
- **Parte 2 — Implementación:** construyes el sistema basándote en tu diseño

---

## El Problema: Asistente de Análisis de Noticias

Una empresa de medios quiere un sistema que, dado un **tema de actualidad**, produzca:

1. 📰 Un **resumen informativo** con los hechos principales
2. ⚖️ Un **análisis de sesgo** que identifique si la cobertura es neutral o tendenciosa  
3. ❓ Una lista de **preguntas sin responder** que el periodismo debería seguir investigando
4. 📋 Un **reporte final** que integre los tres puntos anteriores

**Restricciones del sistema:**
- Los pasos 1, 2 y 3 son **independientes entre sí** — no necesitan el resultado del otro para ejecutarse
- El paso 4 **necesita** los resultados de los pasos 1, 2 y 3
- Cada paso tiene una responsabilidad muy específica (no mezclar tareas)

---
## ⚙️ Setup (no modificar)

In [ ]:
%pip install openai ddgs python-dotenv --quiet

In [ ]:
import os
import json
import time
import concurrent.futures
from openai import OpenAI
from dotenv import load_dotenv
from ddgs import DDGS

load_dotenv()

BACKEND = "nvidia"   # "ollama" | "anthropic" | "openai" | "groq" | "nvidia"

CONFIGS = {
    "ollama":    {"base_url": "http://localhost:11434/v1",          "api_key": "ollama",                           "model": "qwen2.5:7b"},
    "anthropic": {"base_url": "https://api.anthropic.com/v1",      "api_key": os.getenv("ANTHROPIC_API_KEY", ""), "model": "claude-3-5-haiku-20241022"},
    "openai":    {"base_url": None,                                 "api_key": os.getenv("OPENAI_API_KEY", ""),   "model": "gpt-4o-mini"},
    "groq":      {"base_url": "https://api.groq.com/openai/v1",    "api_key": os.getenv("GROQ_API_KEY", ""),     "model": "llama-3.3-70b-versatile"},
    "nvidia":    {"base_url": "https://integrate.api.nvidia.com/v1","api_key": os.getenv("NVIDIA_API_KEY", ""),  "model": "meta/llama-3.3-70b-instruct"},
}

cfg = CONFIGS[BACKEND]
client_kwargs = {"api_key": cfg["api_key"]}
if cfg["base_url"]:
    client_kwargs["base_url"] = cfg["base_url"]

client = OpenAI(**client_kwargs)
MODEL  = cfg["model"]
print(f"✅ Backend: {BACKEND.upper()} | Modelo: {MODEL}")

In [ ]:
# ── Primitivos disponibles (los mismos del notebook anterior) ─────────────────

def web_search(query: str) -> str:
    """Busca información actualizada en la web."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No se encontraron resultados."
        return "\n\n".join(f"**{r['title']}**\n{r['body']}" for r in results)
    except Exception as e:
        return f"Error al buscar: {e}"


SEARCH_REGISTRY = {"web_search": web_search}

SEARCH_TOOL = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Busca información actualizada en la web.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Consulta de búsqueda."}
            },
            "required": ["query"]
        }
    }
}]


def run_agent(
    user_message: str,
    system: str,
    tools: list = None,
    tool_registry: dict = None,
    verbose: bool = True,
    label: str = "Agente",
) -> str:
    """Ejecuta el agentic loop. Retorna el texto final de respuesta."""
    tools         = tools         or []
    tool_registry = tool_registry or {}

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    if verbose:
        preview = user_message[:100] + "..." if len(user_message) > 100 else user_message
        print(f"  [{label}] ← {preview}")

    for _ in range(10):
        kwargs = {"model": MODEL, "messages": messages}
        if tools:
            kwargs["tools"]       = tools
            kwargs["tool_choice"] = "auto"

        response      = client.chat.completions.create(**kwargs)
        choice        = response.choices[0]
        finish_reason = choice.finish_reason

        if finish_reason == "stop":
            text = choice.message.content or ""
            if verbose:
                preview = text[:150] + "..." if len(text) > 150 else text
                print(f"  [{label}] → {preview}")
            return text

        if finish_reason == "tool_calls":
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                args   = json.loads(tc.function.arguments)
                fn     = tool_registry.get(tc.function.name)
                result = fn(**args) if fn else f"Tool '{tc.function.name}' no encontrada."
                if verbose:
                    print(f"  [{label}] 🔧 {tc.function.name}({args})")
                messages.append({
                    "role": "tool", "tool_call_id": tc.id, "content": result
                })
        else:
            break

    return "[Límite de iteraciones alcanzado]"


print("✅ Primitivos listos.")

---
## PARTE 1: Diseño de la Arquitectura

Antes de escribir una sola línea de código, responde las siguientes preguntas.
Edita las celdas de texto directamente.

> 💡 **Por qué importa esto:** elegir el patrón equivocado tiene costos reales.
> Un pipeline secuencial donde los pasos son independientes es innecesariamente lento.
> Un sistema paralelo donde los pasos dependen entre sí produce resultados incorrectos.

### Pregunta 1.1 — Identificación de pasos

Lista los agentes que necesita el sistema. Para cada uno especifica:
- **Nombre** del agente
- **Input** que recibe
- **Output** que produce
- **Necesita web_search?** Sí / No

---
*✏️ Tu respuesta aquí:*

| Agente | Input | Output | web_search |
|--------|-------|--------|------------|
| ... | ... | ... | ... |

### Pregunta 1.2 — Dependencias entre pasos

¿Cuáles pasos pueden correr al mismo tiempo? ¿Cuáles deben esperar a que otro termine?
Dibuja el grafo de dependencias usando texto o ASCII.

Ejemplo de formato:
```
[A] ──┐
[B] ──┼──► [D]
[C] ──┘
```

---
*✏️ Tu respuesta aquí:*

```
...
```

### Pregunta 1.3 — Elección de patrón

¿Qué patrón (o combinación de patrones) usarías para implementar este sistema?
Justifica tu elección en 2-3 oraciones.

Opciones: `Sequential` | `Parallel` | `Orquestador LLM` | `Loop` | **combinación**

---
*✏️ Tu respuesta aquí:*


### Pregunta 1.4 — Riesgos del diseño

Nombra **un riesgo** de tu arquitectura propuesta y cómo lo mitigarías.

Ejemplos de riesgos: latencia, un agente que falla y bloquea todo el sistema,
resultados inconsistentes, costo de tokens, etc.

---
*✏️ Tu respuesta aquí:*


---
## 🔨 PARTE 2: Implementación

Implementa el sistema usando `run_agent` como primitivo.
Los `# TODO` marcan exactamente qué debes completar.

**Regla:** no puedes modificar `run_agent` ni el setup. Solo completas los TODOs.

### 2.1 — Agentes especializados

Cada agente tiene una sola responsabilidad. El reto está en el **system prompt**:
instrucciones precisas producen outputs estructurados que el reporte final puede integrar.

In [ ]:
def summarizer_agent(topic: str, verbose: bool = True) -> str:
    """
    Busca noticias sobre el tema y produce un resumen con los hechos principales.
    Debe usar web_search para obtener información actual.
    Output esperado: 3-5 puntos con los hechos más relevantes.
    """
    # TODO 1: Define el system prompt para este agente.
    # Debe instruir al agente para:
    #   - Buscar información actual con web_search
    #   - Extraer solo hechos verificables, sin opiniones
    #   - Presentar el resultado como 3-5 puntos concisos
    system = """
    ...
    """

    return run_agent(
        user_message=f"Resume las noticias más recientes sobre: {topic}",
        system=system,
        tools=SEARCH_TOOL,
        tool_registry=SEARCH_REGISTRY,
        verbose=verbose,
        label="Summarizer",
    )


def bias_analyst_agent(topic: str, verbose: bool = True) -> str:
    """
    Analiza si la cobertura mediática del tema presenta sesgos.
    Output esperado: identificación de tipo de sesgo (si existe) con ejemplos.
    """
    # TODO 2: Define el system prompt para este agente.
    # Debe instruir al agente para:
    #   - Buscar cobertura de diferentes fuentes con web_search
    #   - Identificar si hay lenguaje cargado, omisión de perspectivas, etc.
    #   - Ser objetivo — si no hay sesgo evidente, decirlo explícitamente
    system = """
    ...
    """

    return run_agent(
        user_message=f"Analiza el sesgo en la cobertura mediática de: {topic}",
        system=system,
        tools=SEARCH_TOOL,
        tool_registry=SEARCH_REGISTRY,
        verbose=verbose,
        label="BiasAnalyst",
    )


def questions_agent(topic: str, verbose: bool = True) -> str:
    """
    Genera preguntas periodísticas que quedan sin responder en la cobertura actual.
    Output esperado: 3-5 preguntas concretas y relevantes.
    """
    # TODO 3: Define el system prompt para este agente.
    # Debe instruir al agente para:
    #   - Buscar qué se ha cubierto sobre el tema
    #   - Identificar vacíos informativos (qué no se ha investigado)
    #   - Formular preguntas específicas y accionables, no genéricas
    system = """
    ...
    """

    return run_agent(
        user_message=f"¿Qué preguntas quedan sin responder sobre: {topic}?",
        system=system,
        tools=SEARCH_TOOL,
        tool_registry=SEARCH_REGISTRY,
        verbose=verbose,
        label="QuestionsAgent",
    )


print("✅ Agentes especializados definidos.")

### 2.2 — Agente integrador

El agente de reporte recibe los outputs de los tres agentes anteriores
y los integra en un reporte cohesivo. **No usa web_search** — solo procesa texto.

In [ ]:
def report_agent(summary: str, bias: str, questions: str, topic: str, verbose: bool = True) -> str:
    """
    Integra los tres análisis en un reporte periodístico final.
    No usa web_search — trabaja solo con los inputs recibidos.
    """
    # TODO 4: Define el system prompt para el agente integrador.
    # Debe instruir al agente para:
    #   - Sintetizar los tres inputs en un reporte estructurado
    #   - Usar secciones claramente diferenciadas
    #   - NO inventar información — solo integrar lo que recibe
    system = """
    ...
    """

    combined_input = f"""
Tema: {topic}

=== RESUMEN DE HECHOS ===
{summary}

=== ANÁLISIS DE SESGO ===
{bias}

=== PREGUNTAS SIN RESPONDER ===
{questions}
"""

    return run_agent(
        user_message=combined_input,
        system=system,
        verbose=verbose,
        label="ReportAgent",
    )


print("✅ Agente integrador definido.")

### 2.3 — Sistema completo

Aquí ensamblas todo. El diseño que propusiste en la Parte 1 se convierte en código.

In [ ]:
def run_news_analysis_system(topic: str, verbose: bool = True) -> dict:
    """
    Sistema completo de análisis de noticias.

    Args:
        topic:   Tema de actualidad a analizar.
        verbose: Si True, muestra el trace de cada agente.

    Returns:
        dict con claves: summary, bias, questions, report, elapsed_s
    """
    print(f"\n{'═'*65}")
    print(f"📰 Analizando: {topic}")
    print(f"{'═'*65}")

    t0 = time.time()

    # TODO 5: Implementa la ejecución de los tres agentes independientes.
    #
    # Pista: revisa tu respuesta en la Pregunta 1.2.
    # Si los tres agentes son independientes, ¿deberían correr en secuencia
    # o en paralelo? Usa concurrent.futures.ThreadPoolExecutor si corresponde.
    #
    # Variables que debes producir:
    #   summary   = resultado de summarizer_agent(topic)
    #   bias      = resultado de bias_analyst_agent(topic)
    #   questions = resultado de questions_agent(topic)

    # --- Tu código aquí ---
    summary   = ...
    bias      = ...
    questions = ...
    # ----------------------

    elapsed_parallel = time.time() - t0
    print(f"\n⏱️  Fase 1 completada en {elapsed_parallel:.1f}s")

    # TODO 6: Llama al agente integrador con los tres resultados.
    print(f"\n{'─'*65}")
    print("📋 Generando reporte final...")

    # --- Tu código aquí ---
    report = ...
    # ----------------------

    elapsed_total = time.time() - t0

    return {
        "summary":   summary,
        "bias":      bias,
        "questions": questions,
        "report":    report,
        "elapsed_s": elapsed_total,
    }


print("✅ Sistema definido. Ejecuta la siguiente celda para probarlo.")

In [ ]:
# 🚀 Ejecutar el sistema
# Cambia el tema por el que quieras analizar

TOPIC = "inteligencia artificial en educación"

result = run_news_analysis_system(TOPIC, verbose=True)

print(f"\n{'═'*65}")
print("REPORTE FINAL")
print(f"{'═'*65}")
print(result["report"])
print(f"\n⏱️  Tiempo total: {result['elapsed_s']:.1f}s")

---
## 🧪 Validaciones

Ejecuta estas celdas para verificar que tu implementación es correcta.

In [ ]:
# Validación 1 — El sistema retorna todos los campos esperados
print("Validación 1: estructura del resultado")
assert isinstance(result, dict), "El resultado debe ser un dict"
for key in ["summary", "bias", "questions", "report", "elapsed_s"]:
    assert key in result, f"Falta la clave '{key}' en el resultado"
    assert result[key] and result[key] != "...", f"'{key}' está vacío o sin implementar"
print("✅ Validación 1 pasada\n")

In [ ]:
# Validación 2 — Cada agente produce output sustancial
print("Validación 2: calidad mínima de cada agente")
assert len(result["summary"].split()) >= 30, (
    f"El resumen parece muy corto ({len(result['summary'].split())} palabras). "
    "Revisa el system prompt del Summarizer."
)
assert len(result["bias"].split()) >= 20, (
    f"El análisis de sesgo parece muy corto. Revisa el system prompt del BiasAnalyst."
)
assert len(result["questions"].split()) >= 20, (
    f"Las preguntas parecen muy cortas. Revisa el system prompt del QuestionsAgent."
)
assert len(result["report"].split()) >= 50, (
    f"El reporte final parece muy corto. Revisa el system prompt del ReportAgent."
)
print(f"  Summarizer:   {len(result['summary'].split())} palabras ✅")
print(f"  BiasAnalyst:  {len(result['bias'].split())} palabras ✅")
print(f"  Questions:    {len(result['questions'].split())} palabras ✅")
print(f"  Report:       {len(result['report'].split())} palabras ✅")
print("✅ Validación 2 pasada\n")

In [ ]:
# Validación 3 — Benchmark de velocidad
# Si implementaste correctamente el paralelismo, los 3 agentes independientes
# deben tardar ~lo mismo que el más lento de los tres, no la suma de los tres.

print("Validación 3: eficiencia del sistema")

# Medir cada agente por separado
import time
t = time.time(); summarizer_agent(TOPIC, verbose=False); t_sum = time.time() - t
t = time.time(); bias_analyst_agent(TOPIC, verbose=False); t_bias = time.time() - t
t = time.time(); questions_agent(TOPIC, verbose=False); t_q = time.time() - t

t_sequential = t_sum + t_bias + t_q
t_actual     = result["elapsed_s"]
t_optimal    = max(t_sum, t_bias, t_q)  # lo mejor posible en paralelo

print(f"  Tiempo secuencial (suma):  {t_sequential:.1f}s")
print(f"  Tiempo óptimo (paralelo):  {t_optimal:.1f}s")
print(f"  Tiempo de tu sistema:      {t_actual:.1f}s")

# Tu sistema debería ser más cercano al óptimo que al secuencial
midpoint = (t_sequential + t_optimal) / 2
if t_actual <= midpoint:
    speedup = t_sequential / t_actual
    print(f"\n✅ Implementación eficiente — speedup ≈ {speedup:.1f}x vs secuencial")
else:
    print(f"\n⚠️  El sistema parece secuencial. ¿Usaste ThreadPoolExecutor en el TODO 5?")
    print(f"   Revisa tu respuesta en la Pregunta 1.2 y el TODO 5.")

---
## Parte 3 (Opcional): Agrega un Loop de Calidad

El sistema actual genera el reporte en **una sola pasada**.
Extiéndelo para que el reporte se auto-evalúe y se refine si no cumple
con un criterio de calidad.

**Criterio sugerido:** el reporte debe mencionar explícitamente los tres componentes
(hechos, sesgo, preguntas pendientes). Si un revisor detecta que falta alguno,
el reporte se regenera.

Pista: agrega un `reviewer_agent` que retorne `APPROVED` o una lista de secciones
faltantes, y un loop `while not approved` alrededor del `report_agent`.

In [1]:
# Tu implementación del loop de calidad (opcional)
pass

---
## Resumen

Si las 3 validaciones pasaron, completa este resumen reflexivo:

### Reflexión final

**¿Coincidió tu diseño (Parte 1) con lo que implementaste (Parte 2)?**
Si hubo diferencias, explica por qué cambiaste de decisión.

*✏️ Tu respuesta:*

---

**¿Cuál fue el agente más difícil de instruir con el system prompt y por qué?**

*✏️ Tu respuesta:*

---

**¿Qué pasaría si el `summarizer_agent` falla (retorna string vacío)?**
¿Tu sistema falla silenciosamente o lo detectas? ¿Cómo lo mejorarías?

*✏️ Tu respuesta:*